# pandas Series — the one-dimensional building block

A `Series` is a list of values **plus a label for each value** (the index).
Every DataFrame column is a Series, and most DataFrame operations return one.

Every section below starts with a tiny example you can check by eye, then (sometimes)
shows the same thing on the real hourly power data.

**What's in here**
- what a Series is: values + index + name
- creating one
- selecting: by label (`loc`), by position (`iloc`), by condition (mask)
- `reindex`: conform to a new set of labels
- index alignment: why `s1 + s2` can produce NaN
- missing values
- everyday methods: `shift`, `diff`, `rolling`, `cumsum`, `rank`, `value_counts`
- `map` vs `replace` vs `apply`
- the `.str` and `.dt` accessors
- combining two Series
- pitfalls

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## 1. What a Series is

Three values, three labels. The labels are the *index*; here we choose them ourselves.

In [2]:
s = pd.Series([100, 200, 300], index=["a", "b", "c"])
s

a    100
b    200
c    300
dtype: int64

The left column (`a b c`) is the index, the right column is the values.
`dtype: int64` says the values are integers.

In [3]:
print("values :", s.values)
print("index  :", s.index)
print("dtype  :", s.dtype)
print("name   :", s.name)          # None: we did not give it a name

values : [100 200 300]
index  : Index(['a', 'b', 'c'], dtype='object')
dtype  : int64
name   : None


If you do not give an index, pandas uses `0, 1, 2, ...` (a `RangeIndex`).

In [4]:
s0 = pd.Series([100, 200, 300])
s0

0    100
1    200
2    300
dtype: int64

A `name` becomes the column name when the Series is put into a DataFrame.

In [5]:
s = pd.Series([100, 200, 300], index=["a", "b", "c"], name="price")
s.to_frame()

,price
a,100
b,200
c,300


## 2. Creating a Series

From a dict: the keys become the index.

In [6]:
pd.Series({"nuclear": 8.0, "ccgt": 12.5, "wind": 6.2})

nuclear     8.0
ccgt       12.5
wind        6.2
dtype: float64

From a scalar plus an index: the value is repeated for every label.

In [7]:
pd.Series(0.0, index=["a", "b", "c"])

a    0.0
b    0.0
c    0.0
dtype: float64

**Pitfall:** a `None` among integers turns the whole Series into floats,
because NaN (the missing marker) only exists for floats.

In [8]:
pd.Series([1, 2, None])

0    1.0
1    2.0
2    NaN
dtype: float64

**Pitfall:** mixing strings and numbers gives `object` dtype. Arithmetic then fails or
does string tricks. Look at the dtype line, always.

In [9]:
mixed = pd.Series([1, "2", 3])
print(mixed.dtype)
mixed

object


0    1
1    2
2    3
dtype: object

## 3. Selecting by label: `loc`

`s.loc[label]` looks up by index label.

In [43]:
s = pd.Series([100, 200, 300], index=["a", "b", "c"])
s.loc["b"]

200

A list of labels returns a smaller Series, in the order you asked for.

In [44]:
s.loc[["c", "a"]]

c    300
a    100
dtype: int64

A label slice is **inclusive at both ends**.

In [45]:
s.loc["a":"c"]
s

a    100
b    200
c    300
dtype: int64

## 4. Selecting by position: `iloc`

`s.iloc[i]` is the i-th value counting from 0, regardless of the labels.
A positional slice is **exclusive at the end**, like Python lists.

In [13]:
print(s.iloc[0])      # first value
print(s.iloc[-1])     # last value
s.iloc[0:2]           # positions 0 and 1, not 2

100
300


a    100
b    200
dtype: int64

**Pitfall:** with an integer index, `s[0]` is ambiguous (label 0 or position 0?).
Here the labels are 10, 20, 30, so `s[0]` is an error while `s.iloc[0]` works.

In [14]:
t = pd.Series([100, 200, 300], index=[10, 20, 30])
try:
    t[0]
except KeyError as e:
    print("KeyError:", e)
print("iloc[0] :", t.iloc[0])
print("loc[10] :", t.loc[10])

KeyError: 0
iloc[0] : 100
loc[10] : 100


## 5. Selecting by condition (boolean mask)

A comparison gives a Series of True/False with the same index.

In [15]:
s = pd.Series([100, 200, 300], index=["a", "b", "c"])
mask = s > 150
mask

a    False
b     True
c     True
dtype: bool

Passing the mask keeps the rows where it is True.

In [16]:
s[mask]

b    200
c    300
dtype: int64

Combine conditions with `&` (and), `|` (or), `~` (not). Each comparison needs parentheses.

In [17]:
s[(s > 100) & (s < 300)]

b    200
dtype: int64

In [18]:
s[s.isin([100, 300])]

a    100
c    300
dtype: int64

## 6. `reindex`: conform to a new set of labels

`reindex` returns a Series with exactly the labels you ask for, in that order.
- a label that exists keeps its value
- a label that does not exist gets NaN
- a label you do not ask for is dropped

In [19]:
s = pd.Series([100, 200, 300], index=[0, 1, 2])
s

0    100
1    200
2    300
dtype: int64

In [20]:
s.reindex([2, 4, 0])

2    300.0
4      NaN
0    100.0
dtype: float64

Label 2 → 300, label 4 → NaN (it never existed), label 0 → 100. Label 1 is gone because
we did not ask for it. Note the dtype changed to float64 because of the NaN.

`fill_value` replaces the NaN for the new labels.

In [21]:
s.reindex([2, 4, 0], fill_value=0)

2    300
4      0
0    100
dtype: int64

This is how you find missing hours in a time series: build the complete grid of
timestamps and reindex to it. Whatever comes out NaN was missing.

In [22]:
have = pd.Series([1.0, 2.0, 4.0],
                 index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00", "2023-01-01 03:00"]))
have

2023-01-01 00:00:00    1.0
2023-01-01 01:00:00    2.0
2023-01-01 03:00:00    4.0
dtype: float64

In [23]:
full_grid = pd.date_range("2023-01-01 00:00", "2023-01-01 04:00", freq="h")
full_grid

DatetimeIndex(['2023-01-01 00:00:00', '2023-01-01 01:00:00', '2023-01-01 02:00:00', '2023-01-01 03:00:00',
               '2023-01-01 04:00:00'],
              dtype='datetime64[ns]', freq='h')

In [24]:
on_grid = have.reindex(full_grid)
on_grid

2023-01-01 00:00:00    1.0
2023-01-01 01:00:00    2.0
2023-01-01 02:00:00    NaN
2023-01-01 03:00:00    4.0
2023-01-01 04:00:00    NaN
Freq: h, dtype: float64

In [25]:
on_grid[on_grid.isna()]        # the missing hours

2023-01-01 02:00:00   NaN
2023-01-01 04:00:00   NaN
dtype: float64

## 7. Index alignment: the most important thing about Series

When you add two Series, pandas matches values **by label**, not by position.

In [26]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])
print(a)
print()
print(b)

x    1
y    2
z    3
dtype: int64

y    10
z    20
w    30
dtype: int64


In [27]:
a + b

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64

- `y`: 2 + 10 = 12 and `z`: 3 + 20 = 23 (labels in both)
- `x` is only in `a`, `w` is only in `b` → NaN
- the result has the union of labels, sorted

If you want missing labels treated as 0, use the method form with `fill_value`.

In [28]:
a.add(b, fill_value=0)

w    30.0
x     1.0
y    12.0
z    23.0
dtype: float64

**Pitfall:** `.values` throws the labels away, so the addition becomes positional.

In [29]:
a.values + b.values      # x+y, y+z, z+w  -> meaningless

array([11, 22, 33])

Same idea when assigning a column back into a DataFrame. A Series is aligned on the
index; a NumPy array is placed by position.

In [30]:
df = pd.DataFrame({"v": [10, 20, 30]}, index=["c", "a", "b"])
extra = pd.Series([1, 2, 3], index=["a", "b", "c"])
df["aligned"] = extra            # matched by label
df["positional"] = extra.values  # placed in order 1, 2, 3
df

,v,aligned,positional
c,10,3,1
a,20,1,2
b,30,2,3


Row `c` got 3 from the aligned Series (label c → 3) but 1 from the array (first position).

**Pitfall:** `==` between Series with different labels raises instead of aligning.

In [31]:
try:
    a == b
except ValueError as e:
    print("ValueError:", e)

ValueError: Can only compare identically-labeled Series objects


## 8. Missing values

`isna()` marks NaN. `== np.nan` never works (NaN is not equal to anything, even itself).

In [32]:
s = pd.Series([1.0, np.nan, 3.0, np.nan], index=["a", "b", "c", "d"])
print(s)
print()
print(s.isna())

a    1.0
b    NaN
c    3.0
d    NaN
dtype: float64

a    False
b     True
c    False
d     True
dtype: bool


In [33]:
print("isna().sum()      :", s.isna().sum())
print("(s == np.nan).sum():", (s == np.nan).sum())    # always 0 -> wrong way

isna().sum()      : 2
(s == np.nan).sum(): 0


Three ways to handle them. Look at what each does to `b` and `d`.

In [34]:
print(s.fillna(0))

a    1.0
b    0.0
c    3.0
d    0.0
dtype: float64


In [35]:
print(s.ffill())          # copy the previous value forward; 'b' gets 1.0, 'd' gets 3.0

a    1.0
b    1.0
c    3.0
d    3.0
dtype: float64


In [36]:
print(s.dropna())

a    1.0
c    3.0
dtype: float64


`mask(condition)` turns values into NaN where the condition is True. Useful for sentinels
like -999.

In [37]:
temp = pd.Series([5.0, -999.0, 7.0])
temp.mask(temp <= -100)

0    5.0
1    NaN
2    7.0
dtype: float64

## 9. Everyday methods

`shift(1)` moves values down one row: each row now holds the previous row's value.
The first row has nothing before it → NaN.

In [38]:
s = pd.Series([10, 20, 30, 40], index=["t1", "t2", "t3", "t4"])
pd.DataFrame({"s": s, "shift(1)": s.shift(1), "shift(-1)": s.shift(-1)})

,s,shift(1),shift(-1)
t1,10,NaN,20.0
t2,20,10.0,30.0
t3,30,20.0,40.0
t4,40,30.0,NaN


`shift(1)` = the past (lag), `shift(-1)` = the future (lead). In a forecasting notebook a
feature uses positive shifts and a target uses a negative shift.

In [39]:
pd.DataFrame({"s": s, "diff()": s.diff(), "pct_change()": s.pct_change(), "cumsum()": s.cumsum()})

,s,diff(),pct_change(),cumsum()
t1,10,NaN,NaN,10
t2,20,10.0,1.000000,30
t3,30,10.0,0.500000,60
t4,40,10.0,0.333333,100


`diff()` is `s - s.shift(1)`; `pct_change()` is `s / s.shift(1) - 1`; `cumsum()` is the running total.

`rolling(2).mean()` averages each value with the one before it. The first row has no
window yet → NaN.

In [40]:
pd.DataFrame({"s": s, "rolling(2).mean()": s.rolling(2).mean()})

,s,rolling(2).mean()
t1,10,NaN
t2,20,15.0
t3,30,25.0
t4,40,35.0


**Pitfall (leakage):** `rolling(2).mean()` at row `t2` uses `t2` itself. If `t2` is the
value you are trying to predict, that is cheating. `shift(1)` first, so the window ends
at the previous row.

In [41]:
pd.DataFrame({
    "s": s,
    "rolling(2).mean()": s.rolling(2).mean(),
    "shift(1).rolling(2).mean()": s.shift(1).rolling(2).mean(),
})

,s,rolling(2).mean(),shift(1).rolling(2).mean()
t1,10,NaN,NaN
t2,20,15.0,NaN
t3,30,25.0,15.0
t4,40,35.0,25.0


`rank()` gives 1 to the smallest; `clip` caps values; `sort_values` reorders (labels move with the values).

In [42]:
u = pd.Series([30, 10, 20], index=["a", "b", "c"])
pd.DataFrame({"u": u, "rank()": u.rank(), "clip(upper=25)": u.clip(upper=25)})

,u,rank(),clip(upper=25)
a,30,3.0,25
b,10,1.0,10
c,20,2.0,20


In [43]:
u.sort_values()

b    10
c    20
a    30
dtype: int64

`value_counts()` counts how often each value appears. `normalize=True` gives shares.
`dropna=False` also counts NaN.

In [44]:
tariff = pd.Series(["Fixed", "TOU", "Fixed", None, "Fixed", "TOU"])
print(tariff.value_counts())
print()
print(tariff.value_counts(dropna=False))
print()
print(tariff.value_counts(normalize=True).round(2))

Fixed    3
TOU      2
Name: count, dtype: int64

Fixed    3
TOU      2
None     1
Name: count, dtype: int64

Fixed    0.6
TOU      0.4
Name: proportion, dtype: float64


Summary statistics. Note pandas `std()` divides by n−1; NumPy divides by n.

In [45]:
v = pd.Series([2.0, 4.0, 6.0])
print("mean       :", v.mean())
print("std pandas :", round(v.std(), 4))         # ddof=1  -> 2.0
print("std numpy  :", round(np.std(v.values), 4))  # ddof=0  -> 1.633
print("idxmax     :", v.idxmax(), "(label of the max)")

mean       : 4.0
std pandas : 2.0
std numpy  : 1.633
idxmax     : 2 (label of the max)


## 10. `map` vs `replace` vs `apply`

`map(dict)`: look every value up in the dict. Values **not in the dict become NaN**.

In [46]:
region = pd.Series(["London", "Wales", "North"])
lookup = {"London": "South", "Wales": "West"}
region.map(lookup)

0    South
1     West
2      NaN
dtype: object

`replace(dict)`: same lookup, but values not in the dict are **kept**.

In [47]:
region.replace(lookup)

0    South
1     West
2    North
dtype: object

`apply(function)`: call a Python function on each value. Works, but it is a slow loop;
prefer the vectorised form when one exists.

In [48]:
p = pd.Series([100, 200, 300])
print(p.apply(lambda x: x * 1.1))
print()
print(p * 1.1)                        # same result, vectorised

0    110.0
1    220.0
2    330.0
dtype: float64

0    110.0
1    220.0
2    330.0
dtype: float64


Converting text that should be numbers: `astype(float)` fails on the first bad value,
`pd.to_numeric(errors="coerce")` turns bad values into NaN so you can count them.

In [49]:
txt = pd.Series(["70.16", "23.19", "missing", "115.79"])
try:
    txt.astype(float)
except ValueError as e:
    print("astype fails:", e)
num = pd.to_numeric(txt, errors="coerce")
print(num)
print("bad values:", num.isna().sum())

astype fails: could not convert string to float: 'missing'
0     70.16
1     23.19
2       NaN
3    115.79
dtype: float64
bad values: 1


## 11. Accessors: `.str` and `.dt`

String methods live under `.str` and work on every element at once.

In [50]:
names = pd.Series(["London ", "wales", "NORTH"])
pd.DataFrame({"raw": names, "strip": names.str.strip(), "lower": names.str.lower(), "title": names.str.strip().str.title()})

,raw,strip,lower,title
0,London,London,london,London
1,wales,wales,wales,Wales
2,NORTH,NORTH,north,North


Date parts live under `.dt`, but only once the values are real timestamps.

In [51]:
times = pd.Series(["2023-01-01 07:00", "2023-01-01 18:00", "2023-01-07 18:00"])
print(times.dtype)                      # object: still strings
try:
    times.dt.hour
except AttributeError as e:
    print("AttributeError:", e)

object
AttributeError: Can only use .dt accessor with datetimelike values


In [52]:
times = pd.to_datetime(times)
print(times.dtype)
pd.DataFrame({"time": times, "hour": times.dt.hour, "dayofweek": times.dt.dayofweek})

datetime64[ns]


,time,hour,dayofweek
0,2023-01-01 07:00:00,7,6
1,2023-01-01 18:00:00,18,6
2,2023-01-07 18:00:00,18,5


(dayofweek: Monday = 0 … Sunday = 6. 2023-01-01 was a Sunday, 2023-01-07 a Saturday.)

## 12. Combining two Series

`pd.concat` stacks them end to end.

In [53]:
s1 = pd.Series([1, 2], index=["a", "b"])
s2 = pd.Series([3], index=["c"])
pd.concat([s1, s2])

a    1
b    2
c    3
dtype: int64

`concat(axis=1)` puts them side by side as columns, aligned by label.

In [54]:
s3 = pd.Series([10, 30], index=["a", "c"])
pd.concat([s1.rename("s1"), s3.rename("s3")], axis=1)

,s1,s3
a,1.0,10.0
b,2.0,NaN
c,NaN,30.0


`combine_first`: fill the NaN in one Series from another (patching a gap from a backup source).

In [55]:
primary = pd.Series([1.0, np.nan, 3.0], index=["a", "b", "c"])
backup = pd.Series([9.0, 2.0, 9.0], index=["a", "b", "c"])
primary.combine_first(backup)

a    1.0
b    2.0
c    3.0
dtype: float64

Only `b` was taken from the backup; `a` and `c` kept their own values.

## 13. Duplicate labels

An index does not have to be unique. When it is not, lookups return several rows and
`reindex` refuses to work. Check `index.is_unique` before any time-series operation.

In [56]:
d = pd.Series([1, 2, 3], index=["a", "a", "b"])
print("is_unique:", d.index.is_unique)
print(d.loc["a"])                      # two rows come back

is_unique: False
a    1
a    2
dtype: int64


In [57]:
try:
    d.reindex(["a", "b", "c"])
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot reindex on an axis with duplicate labels


In [58]:
d[~d.index.duplicated(keep="last")]    # keep the last of each label

a    2
b    3
dtype: int64

## 14. On real data

The same operations on the hourly power file. One Series: consumption indexed by time.

In [59]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df.set_index("time")["consumption_mwh"]
cons.head()

time
2022-01-01 00:00:00+00:00    26858.4
2022-01-01 01:00:00+00:00    26177.8
2022-01-01 02:00:00+00:00    26229.4
2022-01-01 03:00:00+00:00    25381.3
2022-01-01 04:00:00+00:00    25223.0
Name: consumption_mwh, dtype: float64

In [60]:
print("length     :", len(cons))
print("is_unique  :", cons.index.is_unique)
print("sorted     :", cons.index.is_monotonic_increasing)
print("first/last :", cons.index.min(), "->", cons.index.max())

length     : 17520
is_unique  : True
sorted     : True
first/last : 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:00:00+00:00


Select one day by partial date string, then the 18:00 value with `loc`.

In [61]:
one_day = cons.loc["2023-01-24"]
print(len(one_day), "hours")
one_day.loc["2023-01-24 18:00"]

24 hours


39849.6

Lag and rolling features, printed side by side for a few rows so you can check them.

In [62]:
pd.DataFrame({
    "cons": one_day,
    "lag1": one_day.shift(1),
    "roll3_shifted": one_day.shift(1).rolling(3).mean(),
}).head(6)

,cons,lag1,roll3_shifted
time,,,
2023-01-24 00:00:00+00:00,31399.6,NaN,NaN
2023-01-24 01:00:00+00:00,29679.8,31399.6,NaN
2023-01-24 02:00:00+00:00,29344.0,29679.8,NaN
2023-01-24 03:00:00+00:00,29118.6,29344.0,30141.133333
2023-01-24 04:00:00+00:00,28228.2,29118.6,29380.800000
2023-01-24 05:00:00+00:00,29088.6,28228.2,28896.933333


The `roll3_shifted` value at 03:00 is the mean of 00:00, 01:00, 02:00 — check:

In [63]:
print(one_day.iloc[0:3].mean())

30141.13333333333


Reindex to the full hourly grid on the messy raw file to count missing hours.

In [64]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw_cons = raw.set_index(pd.to_datetime(raw["time"], utc=True))["consumption_mwh"]
raw_cons = raw_cons[~raw_cons.index.duplicated()].sort_index()   # dedupe first
grid = pd.date_range(raw_cons.index.min(), raw_cons.index.max(), freq="h")
on_grid = raw_cons.reindex(grid)
print("rows in file :", len(raw_cons))
print("grid hours   :", len(grid))
print("missing      :", on_grid.isna().sum())
on_grid[on_grid.isna()].head()

rows in file : 17442
grid hours   : 17520
missing      : 78


2022-01-07 10:00:00+00:00   NaN
2022-01-17 14:00:00+00:00   NaN
2022-01-28 00:00:00+00:00   NaN
2022-03-11 03:00:00+00:00   NaN
2022-03-27 00:00:00+00:00   NaN
Name: consumption_mwh, dtype: float64

## Quick reference

| Want | Write |
|---|---|
| first / last element | `s.iloc[0]`, `s.iloc[-1]` |
| by label, inclusive slice | `s.loc["a":"c"]` |
| by condition | `s[(s > 1) & (s < 5)]` |
| conform to labels | `s.reindex(labels, fill_value=0)` |
| add with alignment, NaN → 0 | `s1.add(s2, fill_value=0)` |
| previous value / next value | `s.shift(1)` / `s.shift(-1)` |
| lagged rolling mean | `s.shift(1).rolling(24).mean()` |
| sentinel → NaN | `s.mask(s <= -100)` |
| count values | `s.value_counts(dropna=False)` |
| lookup table | `s.map(d)` (unmapped → NaN) or `s.replace(d)` (kept) |
| text → numbers | `pd.to_numeric(s, errors="coerce")` |
| patch gaps from another source | `s1.combine_first(s2)` |
| check before shift/rolling | `s.index.is_unique`, `s.index.is_monotonic_increasing` |